# Fase 4: Addestramento e Ottimizzazione dei Modelli di Machine Learning

In questo notebook si addestrano **5 diversi modelli**:
1. **Logistic Regression** (con ottimizzazione degli iperparametri via RandomizedSearchCV)
2. **Random Forest** (con ottimizzazione degli iperparametri via RandomizedSearchCV)
3. **XGBoost** (con ottimizzazione degli iperparametri via RandomizedSearchCV)
4. **LightGBM** (addestrato direttamente con parametri di default per essere leggero)
5. **Support Vector Machine (SVM)** (addestrato direttamente con parametri di default per essere leggero)

L'addestramento viene ripetuto per ciascuna delle **8 combinazioni di dataset** (2 modalità di scala × 4 modalità di bilanciamento/riduzione). Tutti i modelli ottimizzati vengono salvati in formato `.joblib` nella rispettiva cartella per essere confrontati successivamente.

In [1]:
# Importiamo i moduli e le metriche necessarie per l'addestramento e la valutazione dei modelli
import os
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.svm import SVC

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import loguniform, randint
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, make_scorer
)

In [2]:
# Impostazione dello stato di randomizzazione fisso ed il tipo di punteggio per l'ottimizzazione
STATO_CASUALE = 34
METRICA_SELEZIONE = "f1"  # "f1", "balanced_accuracy", "recall", o "custom"

# Funzione per calcolare un punteggio medico combinato tra Recall e Balanced Accuracy
def calcola_punteggio_medico(y_vero, y_predetto):
    recall = recall_score(y_vero, y_predetto, zero_division=0)
    bilanciata = balanced_accuracy_score(y_vero, y_predetto)
    punteggio_finale = (0.5 * recall) + (0.5 * bilanciata)
    return punteggio_finale

if METRICA_SELEZIONE == "custom":
    misuratore_punteggio = make_scorer(calcola_punteggio_medico, greater_is_better=True)
else:
    misuratore_punteggio = METRICA_SELEZIONE

In [3]:
# Definiamo le cartelle ed i percorsi per l'addestramento
CARTELLA_TRAINING = "../Training"
modalita_scala = ["Clean", "Normalized"]
sotto_cartelle = ["Normal", "Aug", "Pca", "Aug+Pca"]

# Griglia dei pesi per compensare lo sbilanciamento di classe
pesi_classi_griglia = [
    'balanced',
    {0: 1.0, 1: 2.0},
    {0: 1.0, 1: 3.0},
    {0: 1.0, 1: 5.0},
    {0: 1.0, 1: 7.5},
    {0: 1.0, 1: 10.0}
]

# Configurazione delle impostazioni per ciascun modello
configurazioni_modelli = {
    'Logistic Regression': {
        'modello': LogisticRegression(max_iter=10000, random_state=STATO_CASUALE),
        'parametri': {
            'C': loguniform(1e-4, 1e2),
            'class_weight': pesi_classi_griglia
        }
    },
    'Random Forest': {
        'modello': RandomForestClassifier(random_state=STATO_CASUALE),
        'parametri': {
            'class_weight': pesi_classi_griglia,
            'n_estimators': randint(10, 200),
            'max_depth': [3, 5, 10, 15, None],
            'min_samples_split': randint(2, 11),
            'min_samples_leaf': randint(1, 11)
        }
    },
    'XGBoost': {
        'modello': xgb.XGBClassifier(random_state=STATO_CASUALE, eval_metric='logloss'),
        'parametri': {
            'n_estimators': randint(10, 200),
            'max_depth': [3, 5, 7, 9],
            'learning_rate': loguniform(1e-3, 1e-1),
            'scale_pos_weight': [1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
        }
    },
    'LightGBM': {
        'modello': lgb.LGBMClassifier(random_state=STATO_CASUALE, verbosity=-1, class_weight='balanced')
    },
    'SVM': {
        'modello': SVC(random_state=STATO_CASUALE, probability=True, class_weight='balanced')
    }
}

report_valutazioni = []

# Cicli espliciti per addestrare tutti i modelli su tutte le cartelle
for scala in modalita_scala:
    for cartella in sotto_cartelle:
        percorso_dati = os.path.join(CARTELLA_TRAINING, scala, cartella)
        print(f"\n==================== INIZIO ADDESTRAMENTO: {scala} / {cartella} ====================")
        
        try:
            # Caricamento dei dati di train e validation
            X_train = pd.read_csv(os.path.join(percorso_dati, "X_train.csv"))
            X_val = pd.read_csv(os.path.join(percorso_dati, "X_val.csv"))
            y_train = pd.read_csv(os.path.join(percorso_dati, "Y_train.csv")).squeeze().values.ravel()
            y_val = pd.read_csv(os.path.join(percorso_dati, "Y_val.csv")).squeeze().values.ravel()
            print(f"📦 Caricati correttamente i dati da {percorso_dati}")
        except Exception as e:
            print(f"⚠️ Errore durante il caricamento dei dati: {e}")
            continue
            
        strategia_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=STATO_CASUALE)
        
        for nome_modello, config in configurazioni_modelli.items():
            # Se sono definiti i parametri, eseguiamo la ricerca iperparametri (tuning)
            if 'parametri' in config:
                print(f"🚀 Ottimizzazione iperparametri con tuning per {nome_modello}...")
                ricerca_iperparametri = RandomizedSearchCV(
                    estimator=config['modello'], 
                    param_distributions=config['parametri'], 
                    n_iter=15,
                    cv=strategia_kfold, 
                    scoring=misuratore_punteggio, 
                    n_jobs=-1,
                    random_state=STATO_CASUALE
                )
                ricerca_iperparametri.fit(X_train, y_train)
                modello_addestrato = ricerca_iperparametri.best_estimator_
                parametri_salvati = str(ricerca_iperparametri.best_params_)
            else:
                # Altrimenti, addestriamo direttamente il modello in modo leggero e veloce
                print(f"🚀 Addestramento leggero senza tuning per {nome_modello}...")
                modello_addestrato = config['modello']
                modello_addestrato.fit(X_train, y_train)
                parametri_salvati = "Default (Nessun tuning)"
            
            # Salvataggio del modello addestrato nella sottocartella specifica
            nome_file_salvataggio = nome_modello.lower().replace(" ", "_") + ".joblib"
            percorso_salvataggio = os.path.join(percorso_dati, nome_file_salvataggio)
            joblib.dump(modello_addestrato, percorso_salvataggio)
            print(f"💾 Modello salvato in: {percorso_salvataggio}")
            
            # Calcolo delle predizioni e le metriche sul validation set
            y_predetto = modello_addestrato.predict(X_val)
            try:
                y_probabilita = modello_addestrato.predict_proba(X_val)[:, 1]
                auc_score = roc_auc_score(y_val, y_probabilita)
            except Exception:
                auc_score = np.nan
                
            acc = accuracy_score(y_val, y_predetto)
            bal_acc = balanced_accuracy_score(y_val, y_predetto)
            prec = precision_score(y_val, y_predetto, zero_division=0)
            rec = recall_score(y_val, y_predetto)
            f1 = f1_score(y_val, y_predetto)
            
            report_valutazioni.append({
                'ScaleMode': scala,
                'Subset': cartella,
                'Modello': nome_modello,
                'Accuracy': acc,
                'Balanced_Accuracy': bal_acc,
                'Precision': prec,
                'Recall': rec,
                'F1-Score': f1,
                'ROC-AUC': auc_score,
                'Best_Params': parametri_salvati
            })

# Salviamo il file riassuntivo delle performance di tutti i modelli
df_report_valutazioni = pd.DataFrame(report_valutazioni)
percorso_csv_report = os.path.join(CARTELLA_TRAINING, "model_evaluation_results.csv")
df_report_valutazioni.to_csv(percorso_csv_report, index=False)
print(f"\n🎉 Addestramento e valutazione completati per tutte le 40 combinazioni! Risultati salvati in: {percorso_csv_report}")


==================== INIZIO ADDESTRAMENTO: Clean / Normal ====================
📦 Caricati correttamente i dati da ../Training/Clean/Normal
🚀 Ottimizzazione iperparametri con tuning per Logistic Regression...
💾 Modello salvato in: ../Training/Clean/Normal/logistic_regression.joblib
🚀 Ottimizzazione iperparametri con tuning per Random Forest...
💾 Modello salvato in: ../Training/Clean/Normal/random_forest.joblib
🚀 Ottimizzazione iperparametri con tuning per XGBoost...
💾 Modello salvato in: ../Training/Clean/Normal/xgboost.joblib
🚀 Addestramento leggero senza tuning per LightGBM...
💾 Modello salvato in: ../Training/Clean/Normal/lightgbm.joblib
🚀 Addestramento leggero senza tuning per SVM...
💾 Modello salvato in: ../Training/Clean/Normal/svm.joblib

==================== INIZIO ADDESTRAMENTO: Clean / Aug ====================
📦 Caricati correttamente i dati da ../Training/Clean/Aug
🚀 Ottimizzazione iperparametri con tuning per Logistic Regression...
💾 Modello salvato in: ../Training/Clean/Au